In [4]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root:", project_root)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root: c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project


In [6]:
from src.crawl.pipeline import crawl_seed_urls_and_save_manifest

records = crawl_seed_urls_and_save_manifest(
    seed_file_path="seed_urls.txt",
    manifest_output_path="data/processed/crawl_manifest.csv",
)

for r in records:
    print(r["page_id"], "|", r["extraction_status"], "|", r["url"])

1 | ok | https://en.wikipedia.org/wiki/2024_Formula_One_World_Championship
2 | ok | https://en.wikipedia.org/wiki/2023_Formula_One_World_Championship
3 | ok | https://en.wikipedia.org/wiki/Lewis_Hamilton
4 | ok | https://en.wikipedia.org/wiki/Max_Verstappen
5 | ok | https://en.wikipedia.org/wiki/Scuderia_Ferrari
6 | ok | https://en.wikipedia.org/wiki/Mercedes-AMG_Petronas_F1_Team
7 | ok | https://en.wikipedia.org/wiki/McLaren_F1_Team
8 | ok | https://en.wikipedia.org/wiki/2024_Monaco_Grand_Prix
9 | ok | https://en.wikipedia.org/wiki/Formula_One_constructors%27_championship
10 | ok | https://en.wikipedia.org/wiki/List_of_Formula_One_World_Drivers%27_Champions
11 | ok | https://www.formulaonehistory.com/seasons/
12 | ok | https://www.formulaonehistory.com/category/history/grand-prix-winners/


In [7]:
# vérifie le contenu d'une page Wikipedia et une page formulaonehistory
for page_id in [1, 3, 11]:
    with open(f"../data/processed/seed_page_{page_id}.txt", "r", encoding="utf-8") as f:
        content = f.read()
    print(f"\n=== seed_page_{page_id}.txt ({len(content)} chars) ===")
    print(content[:500])
    print("...")


=== seed_page_1.txt (89892 chars) ===
2024 Formula One World Championship
The 2024 FIA Formula One World Championship was a motor racing championship for Formula One cars and was the 75th running of the Formula One World Championship. It was recognised by the Fédération Internationale de l'Automobile (FIA), the governing body of international motorsport, as the highest class of competition for open-wheel racing cars. The championship was contested over a record twenty-four Grands Prix held around the world.
Drivers and teams compete
...

=== seed_page_3.txt (179997 chars) ===
Lewis Hamilton
Lewis Hamilton | |
|---|---|
| Born | Lewis Carl Davidson Hamilton 7 January 1985 Stevenage, Hertfordshire, England |
| Partner | Nicole Scherzinger (2007–2015) |
| Relatives | Nicolas Hamilton (half-brother) |
| Awards | Full list |
| Formula One World Championship career | |
| Nationality | British |
| 2026 team | Ferrari |
| Car number | 44[note 1] |
| Entries | 382 (382 starts) |
| Championship

In [8]:
for page_id in [2, 4, 5, 6, 7, 8, 9, 10]:
    with open(f"../data/processed/seed_page_{page_id}.txt", "r", encoding="utf-8") as f:
        content = f.read()
    # cherche si la page contient des tableaux markdown
    has_table = "| --- |" in content or "|---|---|" in content
    print(f"seed_page_{page_id}.txt | {len(content)} chars | table: {has_table}")

seed_page_2.txt | 93651 chars | table: True
seed_page_4.txt | 171271 chars | table: True
seed_page_5.txt | 46043 chars | table: True
seed_page_6.txt | 60215 chars | table: True
seed_page_7.txt | 151648 chars | table: True
seed_page_8.txt | 13956 chars | table: True
seed_page_9.txt | 12904 chars | table: True
seed_page_10.txt | 19183 chars | table: True


In [9]:
import re

def normalize_text(text: str) -> str:
    # supprime les lignes de tableau markdown (commencent par |)
    text = re.sub(r'^\|.*\|$', '', text, flags=re.MULTILINE)
    # supprime les lignes vides multiples laissées par la suppression
    text = re.sub(r'\n{3,}', '\n\n', text)
    # supprime les espaces multiples
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# test sur seed_page_3.txt
with open("../data/processed/seed_page_3.txt", "r", encoding="utf-8") as f:
    content = f.read()

cleaned = normalize_text(content)
print(cleaned[:1000])

Lewis Hamilton Lewis Hamilton | | Sir Lewis Carl Davidson Hamilton (born 7 January 1985) is a British racing driver who competes in Formula One for Ferrari. Hamilton has won a joint-record seven Formula One World Drivers' Championship titles—tied with Michael Schumacher—and holds the records for most wins (105), pole positions (104), and podium finishes (203), among others. Born and raised in Stevenage, Hamilton began his career in karting at age six, winning several national titles and attracting the attention of Ron Dennis, who signed him to the McLaren-Mercedes Young Driver Programme in 1998. After winning the Karting World Cup and European Championship in 2000, Hamilton progressed to junior formulae, where his successes included winning the Formula 3 Euro Series and the GP2 Series. He subsequently signed for McLaren in 2007, becoming the first black driver to compete in Formula One at the Australian Grand Prix. In his rookie season, Hamilton won four Grands Prix and set several rec